In [ ]:
!pip install pyspark --quiet
print("PySpark Installation Complete")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
spark=SparkSession.builder\
.appName('Day4 BiigData sales')\
.config('spark.sql.adaptive enables','true')\
.getOrCreate()
print(f"Spark version:{spark.version}")
print(f"SparkSession : ACTIVE")

In [ ]:
df_bronze=spark.read\
.option('header','true')\
.option('inferSchema','true')\
.csv('large_sales_data.csv')
print('===BRONZE LAYER-Raw Data ===')
print(f'Rows: {df_bronze.count()}')
print(f'Columns:{len(df_bronze.columns)}')
print(f'Names:{df_bronze.columns}')
print()
df_bronze.printSchema()

In [ ]:
print("First 5 rows:")
df_bronze.show(5,truncate=False)
print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()


In [ ]:
from typing import ClassVar
df_bronze.write\
.mode('overwrite')\
.parquet('sales_bronze.parquet')
print('Bronze parquet saved:sales_bronze.parquet')
import os
def get_dir_size(path):
  """Get total size of a file or directoryin KB"""
  if os.path.isfile(path):
    return os.path.getsize(path) /1024
  total=0
  for dirpath,dirnames,filenames in os.walk(path):
    for f in filenames:
      total+=os.path.getsize(os.path.join(dirpath,f))
  return total/1024

csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction=(1-parquet_size/csv_size)*100
print(f"\nCSV size :{csv_size:.1f}KB")
print(f"Parquet file size :{parquet_size:.1f}KB")
print(f"Reduction :{reduction:.1f}%smaller")
print(f"\nAt 1 TB scale CSV=1000 GB -> parquet={1000*(1-reduction/100):.0f}GB")

In [ ]:
df_silver=df_bronze\
.dropDuplicates()\
.dropna(subset=['order_id','product','revenue'])
df_silver=df_silver.withColumn(
    'order_date',to_date(col('order_date'),'yyy-MM-dd')
)
df_silver=df_silver\
.withColumn('order_year',year(col('order_date')))\
.withColumn('order_month',month(col('order_date')))
df_silver=df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue')>40000,'High')\
    .when(col('revenue')>1000,'Medium')\
    .otherwise('Low')
)
print(f"Silver Layer rows:{df_silver.count()}")
print("New columns added:order_year,order_month,revenue_category")
df_silver.select('product','revenue','order_year','order_month','revenue_category').show()

In [ ]:
df_silver.write\
.mode('overwrite')\
.parquet('sales_silver.parquet')
print("Silver Parquet saved:sales_silver.parquet")
print(f"Silver size:{get_dir_size("sales_silver.parquet"):1f}KB")
df_verify=spark.read.parquet('sales_silver.parquet')
print('\n===Verify Silver layer===')
print(f"Read-back rows:{df_verify.count()} (should match silver count)")
df_verify.printSchema()
